# 02 · Explainable Fuzzy Ensemble Genetic Framework — Model Pipeline

**Prerequisite:** Run `01_data_cleaning.ipynb` first to produce `../data/combined_dataset.csv`.

```
Phase 1  — Dataset loading, EDA, visualisations
Phase 2  — Baseline models  (Decision Tree / Random Forest / XGBoost)
Phase 3  — Fuzzy Logic framework  (scikit-fuzzy)
Phase 4  — Genetic Algorithm optimisation  (DEAP)
Phase 5  — GA-Fusion ensemble validation + ROC + F1
Phase 6  — Explainable AI  (SHAP global + LIME local)
Phase 7  — Interactive user prediction
```

## Phase 1 · Dataset Analysis

### 1.1 Global Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import LabelEncoder, label_binarize
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier, VotingClassifier
from sklearn.metrics          import (accuracy_score, classification_report,
                                      confusion_matrix, roc_curve, auc)
import xgboost as xgb

import skfuzzy as fuzz
from skfuzzy import control as ctrl

from deap import base, creator, tools, algorithms

import shap
from lime import lime_tabular

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.makedirs('../results', exist_ok=True)
print("All imports OK")

### 1.2 Load Dataset

> **Data Leakage Fix:** We use the `split` column written by `01_data_cleaning.ipynb` to preserve the **exact same train/test partition** that was used during SMOTE — preventing any test-data contamination.

In [ ]:
df_full = pd.read_csv("../data/combined_dataset.csv")

# Restore the original split — no reshuffling
train_df = df_full[df_full['split'] == 'train'].drop(columns=['split']).reset_index(drop=True)
test_df  = df_full[df_full['split'] == 'test'].drop(columns=['split']).reset_index(drop=True)

# Full df (without split col) for EDA and fuzzy rule calibration
df = df_full.drop(columns=['split'])

print("Total rows :", df.shape[0])
print("Train rows :", train_df.shape[0])
print("Test rows  :", test_df.shape[0])
print("Columns    :", df.columns.tolist())
display(df.head())

### 1.3 Engineered Feature Verification & Stats

In [ ]:
print("=== ENGINEERED FEATURES ===")
for feat in ['n_p_ratio', 'n_k_ratio', 'temp_rainfall']:
    status = 'FOUND' if feat in df.columns else 'MISSING'
    print(" ", feat, ":", status)

print()
print("Numeric statistics:")
display(df.describe().round(2))

### 1.4 Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df.select_dtypes(include='number').corr(),
    annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax
)
ax.set_title('Figure 1: Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/Figure1_Correlation_Heatmap.png', dpi=300)
plt.show()

### 1.5 Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
crop_counts = df['crop'].value_counts()
sns.barplot(x=crop_counts.index, y=crop_counts.values, palette='viridis', ax=ax)
ax.set_title('Figure 2: Crop Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Crop'); ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig('../results/Figure2_Class_Distribution.png', dpi=300)
plt.show()

## Phase 2 · Baseline Models

Establishes the 'before' performance before the fuzzy+GA framework.

In [ ]:
le = LabelEncoder()
le.fit(df['crop'])  # fit on all known classes

X_train = train_df.drop(columns=['crop'])
y_train = le.transform(train_df['crop'])
X_test  = test_df.drop(columns=['crop'])
y_test  = le.transform(test_df['crop'])

# Convenience reference (used by fuzzy rule calibration)
X = df.drop(columns=['crop'])

print("X_train:", X_train.shape, " | X_test:", X_test.shape)
print("Classes :", len(le.classes_), "—", list(le.classes_[:5]), "...")

In [ ]:
dt      = DecisionTreeClassifier(max_depth=15, random_state=RANDOM_STATE)
rf      = RandomForestClassifier(n_estimators=100, max_depth=20,
                                  random_state=RANDOM_STATE, n_jobs=-1)
xgb_clf = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=RANDOM_STATE, n_jobs=-1
)

for name, clf in [('Decision Tree', dt), ('Random Forest', rf), ('XGBoost', xgb_clf)]:
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(name.ljust(20), " accuracy:", round(acc, 4))

dt_preds  = dt.predict(X_test)
rf_preds  = rf.predict(X_test)
xgb_preds = xgb_clf.predict(X_test)
dt_acc    = accuracy_score(y_test, dt_preds)
rf_acc    = accuracy_score(y_test, rf_preds)
xgb_acc   = accuracy_score(y_test, xgb_preds)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Decision Tree', 'Random Forest', 'XGBoost'],
              [dt_acc*100, rf_acc*100, xgb_acc*100],
              color=['#95a5a6','#7f8c8d','#2980b9'])
ax.bar_label(bars, fmt='%.2f%%', padding=3)
ax.set_ylim(0, 107); ax.set_ylabel('Accuracy (%)')
ax.set_title('Figure 3: Baseline Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/Figure3_Baseline_Accuracy.png', dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
cm_xgb = confusion_matrix(y_test, xgb_preds)
sns.heatmap(cm_xgb, annot=False, cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Figure 4: XGBoost Baseline Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('../results/Figure4_Baseline_CM.png', dpi=300)
plt.show()

## Phase 3 · Fuzzy Logic Framework

Broadened membership functions for N, P, K, pH, Rainfall, Temperature covering full ML decision space.

In [ ]:
# CELL 1: SYNCHRONIZED FUZZY ARCHITECTURE (ADJUSTED OVERLAP)
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

print("--- PHASE 3: OPTIMIZING INPUT COVERAGE FOR ML ALIGNMENT ---")

# Define Universes
range_N = np.arange(0, 151, 1)
range_P = np.arange(0, 151, 1)
range_K = np.arange(0, 201, 1)
range_pH = np.arange(3.5, 10.1, 0.1)
range_Rain = np.arange(0, 3001, 10)
range_Temp = np.arange(0, 51, 0.5)
range_Crop = np.arange(0, 40, 0.05)

# Create Antecedents and Consequent
N_fuzzy = ctrl.Antecedent(range_N, 'Nitrogen')
P_fuzzy = ctrl.Antecedent(range_P, 'Phosphorus')
K_fuzzy = ctrl.Antecedent(range_K, 'Potassium')
pH_fuzzy = ctrl.Antecedent(range_pH, 'pH')
Rainfall_fuzzy = ctrl.Antecedent(range_Rain, 'Rainfall')
Temperature_fuzzy = ctrl.Antecedent(range_Temp, 'Temperature')
Crop_fuzzy = ctrl.Consequent(range_Crop, 'Crop')

# Optimized Input MFs with higher overlap to capture ML variances
# Broadening the 'Medium' range to capture more samples
N_fuzzy['Low'] = fuzz.trimf(range_N, [0, 0, 80])
N_fuzzy['Medium'] = fuzz.trimf(range_N, [20, 75, 130])
N_fuzzy['High'] = fuzz.trimf(range_N, [70, 150, 150])

P_fuzzy['Low'] = fuzz.trimf(range_P, [0, 0, 80])
P_fuzzy['Medium'] = fuzz.trimf(range_P, [20, 75, 130])
P_fuzzy['High'] = fuzz.trimf(range_P, [70, 150, 150])

K_fuzzy['Low'] = fuzz.trimf(range_K, [0, 0, 110])
K_fuzzy['Medium'] = fuzz.trimf(range_K, [30, 100, 170])
K_fuzzy['High'] = fuzz.trimf(range_K, [90, 200, 200])

pH_fuzzy['Acidic'] = fuzz.trimf(range_pH, [3.5, 3.5, 6.5])
pH_fuzzy['Neutral'] = fuzz.trimf(range_pH, [5.5, 7.0, 8.5])
pH_fuzzy['Alkaline'] = fuzz.trimf(range_pH, [7.5, 10.0, 10.0])

Rainfall_fuzzy.automf(3, names=['Low', 'Medium', 'High'])
Temperature_fuzzy.automf(3, names=['Cold', 'Optimal', 'Hot'])

# Keep the synchronized Gaussian outputs
crop_names = le.classes_
for i, crop_name in enumerate(crop_names):
    Crop_fuzzy[crop_name] = fuzz.gaussmf(Crop_fuzzy.universe, float(i), 0.4)

print("✅ Input Membership Functions broadened for higher ML decision coverage.")

In [ ]:
# CELL 2: GENERATING THE UPDATED FUZZY MEMBERSHIP GRID (GRAPHIC 5)
import matplotlib.pyplot as plt
import seaborn as sns
import skfuzzy as fuzz

sns.set_theme(style="whitegrid")

# Create a 2x3 grid of subplots for the inputs
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Figure 5: Optimized Input Membership Functions (Higher Overlap)", fontsize=20, fontweight='bold', y=1.02)

# Plot Nitrogen (Using the broadened trimf definitions from Phase 3)
axes[0, 0].plot(range_N, fuzz.trimf(range_N, [0, 0, 80]), 'b', label='Low')
axes[0, 0].plot(range_N, fuzz.trimf(range_N, [20, 75, 130]), 'g', label='Medium')
axes[0, 0].plot(range_N, fuzz.trimf(range_N, [70, 150, 150]), 'r', label='High')
axes[0, 0].set_title("Nitrogen (N) - Broadened")
axes[0, 0].legend()

# Plot Phosphorus
axes[0, 1].plot(range_P, fuzz.trimf(range_P, [0, 0, 80]), 'b', label='Low')
axes[0, 1].plot(range_P, fuzz.trimf(range_P, [20, 75, 130]), 'g', label='Medium')
axes[0, 1].plot(range_P, fuzz.trimf(range_P, [70, 150, 150]), 'r', label='High')
axes[0, 1].set_title("Phosphorus (P) - Broadened")

# Plot Potassium
axes[0, 2].plot(range_K, fuzz.trimf(range_K, [0, 0, 110]), 'b', label='Low')
axes[0, 2].plot(range_K, fuzz.trimf(range_K, [30, 100, 170]), 'g', label='Medium')
axes[0, 2].plot(range_K, fuzz.trimf(range_K, [90, 200, 200]), 'r', label='High')
axes[0, 2].set_title("Potassium (K) - Broadened")

# Plot pH
axes[1, 0].plot(range_pH, fuzz.trimf(range_pH, [3.5, 3.5, 6.5]), 'b', label='Acidic')
axes[1, 0].plot(range_pH, fuzz.trimf(range_pH, [5.5, 7.0, 8.5]), 'g', label='Neutral')
axes[1, 0].plot(range_pH, fuzz.trimf(range_pH, [7.5, 10.0, 10.0]), 'r', label='Alkaline')
axes[1, 0].set_title("Soil pH - Broadened")

# Plot Rainfall
axes[1, 1].plot(range_Rain, fuzz.trimf(range_Rain, [0, 0, 1500]), 'b', label='Low')
axes[1, 1].plot(range_Rain, fuzz.trimf(range_Rain, [0, 1500, 3000]), 'g', label='Medium')
axes[1, 1].plot(range_Rain, fuzz.trimf(range_Rain, [1500, 3000, 3000]), 'r', label='High')
axes[1, 1].set_title("Rainfall (mm)")

# Plot Temperature
axes[1, 2].plot(range_Temp, fuzz.trimf(range_Temp, [0, 0, 25]), 'b', label='Cold')
axes[1, 2].plot(range_Temp, fuzz.trimf(range_Temp, [0, 25, 50]), 'g', label='Optimal')
axes[1, 2].plot(range_Temp, fuzz.trimf(range_Temp, [25, 50, 50]), 'r', label='Hot')
axes[1, 2].set_title("Temperature (°C)")

plt.tight_layout()
plt.savefig('../results/Figure5_Updated_Fuzzy_Grid.png', dpi=300, bbox_inches='tight')
plt.show()

# Visualizing the GAUSSIAN Crop Outputs for Alignment
plt.figure(figsize=(15, 4))
for i, crop in enumerate(le.classes_[:10]):
    plt.plot(range_Crop, fuzz.gaussmf(range_Crop, float(i), 0.4), label=crop)
plt.title("Visual Sample: Synchronized Gaussian Crop Outputs (First 10 Classes)")
plt.xlabel("ML Class Index")
plt.ylabel("Membership Degree")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# CELL 3: FULLY ALIGNED MULTI-DIMENSIONAL RULE BASE
print("--- COMPILING HIGH-ALIGNMENT RULES ---")

crop_means = df.groupby('crop').mean()
crop_names = le.classes_

# Helper for mapping internal state to labels
def get_precise_label(val, antecedent, labels):
    mfs = [fuzz.interp_membership(antecedent.universe, antecedent[l].mf, val) for l in labels]
    return labels[np.argmax(mfs)]

rules = []
for i, crop_name in enumerate(crop_names):
    stats = crop_means.loc[crop_name]

    # Determine best labels for ALL dimensions to ensure maximum alignment with ML data
    n_l = get_precise_label(stats['n'], N_fuzzy, ['Low', 'Medium', 'High'])
    p_l = get_precise_label(stats['p'], P_fuzzy, ['Low', 'Medium', 'High'])
    k_l = get_precise_label(stats['k'], K_fuzzy, ['Low', 'Medium', 'High'])
    ph_l = get_precise_label(stats['ph'], pH_fuzzy, ['Acidic', 'Neutral', 'Alkaline'])
    r_l = get_precise_label(stats['rainfall'], Rainfall_fuzzy, ['Low', 'Medium', 'High'])
    t_l = get_precise_label(stats['temperature'], Temperature_fuzzy, ['Cold', 'Optimal', 'Hot'])

    # Complex multidimensional rule for 1-to-1 mapping
    rule = ctrl.Rule(
        N_fuzzy[n_l] & P_fuzzy[p_l] & K_fuzzy[k_l] &
        pH_fuzzy[ph_l] & Rainfall_fuzzy[r_l] & Temperature_fuzzy[t_l],
        Crop_fuzzy[crop_name]
    )
    rules.append(rule)

crop_ctrl = ctrl.ControlSystem(rules)
crop_simulator = ctrl.ControlSystemSimulation(crop_ctrl)

print(f"✅ Generated {len(rules)} multi-dimensional rules to mimic ML decision space.")

In [ ]:
# FIXED: y_test is int-encoded; fuzzy returns int index — compare correctly
print('--- EVALUATING FUZZY BASELINE ---')

def predict_fuzzy(n, p, k, ph, rain, temp):
    try:
        crop_simulator.input['Nitrogen']    = n
        crop_simulator.input['Phosphorus']  = p
        crop_simulator.input['Potassium']   = k
        crop_simulator.input['pH']          = ph
        crop_simulator.input['Rainfall']    = rain
        crop_simulator.input['Temperature'] = temp
        crop_simulator.compute()
        return int(round(crop_simulator.output['Crop']))
    except:
        return -1

sample_size      = 100
test_samples     = X_test.iloc[:sample_size]
test_labels_int  = y_test[:sample_size]   # already integer-encoded

fuzzy_preds = []
for i in range(sample_size):
    row = test_samples.iloc[i]
    fuzzy_preds.append(predict_fuzzy(
        row['n'], row['p'], row['k'],
        row['ph'], row['rainfall'], row['temperature']
    ))

correct  = sum(1 for f, t in zip(fuzzy_preds, test_labels_int) if f == t)
coverage = sum(1 for f in fuzzy_preds if f != -1)
print("Fuzzy accuracy  :", round(correct/sample_size*100, 2), "%")
print("Fuzzy coverage  :", round(coverage/sample_size*100, 1), "% (non-error predictions)")

In [ ]:
def get_fuzzy_label(value, antecedent, labels=None):
    '''Return the linguistic label with highest membership for value.
    Provide labels explicitly for antecedents that use non-standard names
    (e.g. pH uses Acidic/Neutral/Alkaline instead of Low/Medium/High).
    '''
    if labels is None:
        labels = ['Low', 'Medium', 'High']
    memberships = [
        (lbl, fuzz.interp_membership(antecedent.universe, antecedent[lbl].mf, value))
        for lbl in labels
    ]
    return max(memberships, key=lambda x: x[1])[0]

# FIXED: use a local sample; no dependency on Phase-4 variables
_s = X_test.iloc[0]
n_word  = get_fuzzy_label(_s['n'],        N_fuzzy)
p_word  = get_fuzzy_label(_s['p'],        P_fuzzy)
k_word  = get_fuzzy_label(_s['k'],        K_fuzzy)
ph_word = get_fuzzy_label(_s['ph'],       pH_fuzzy, ['Acidic','Neutral','Alkaline'])
r_word  = get_fuzzy_label(_s['rainfall'], Rainfall_fuzzy)
print("--- FUZZY INTERPRETATION OF TEST SAMPLE [0] ---")
print("N  =", round(_s["n"],1),       "->", n_word)
print("P  =", round(_s["p"],1),       "->", p_word)
print("K  =", round(_s["k"],1),       "->", k_word)
print("pH =", round(_s["ph"],2),      "->", ph_word)
print("Rain =", round(_s["rainfall"],1), "->", r_word)

## Phase 4 · Genetic Algorithm Optimisation

DEAP evolves optimal weights and hyperparameters for the DT + RF + XGBoost fusion ensemble.

In [ ]:
# PHASE 4.1: GENETIC ALGORITHM CONFIGURATION
from deap import base, creator, tools, algorithms
import random
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Setup GA Framework
if "FitnessMax" in creator.__dict__: del creator.FitnessMax
if "Individual" in creator.__dict__: del creator.Individual

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)
toolbox = base.Toolbox()

# 2. Define Hyperparameter Genes
# Genes for Decision Tree, Random Forest, and XGBoost
toolbox.register("dt_depth", random.randint, 10, 50)
toolbox.register("rf_est", random.randint, 50, 300)
toolbox.register("rf_depth", random.randint, 10, 50)
toolbox.register("xgb_est", random.randint, 50, 250)
toolbox.register("xgb_depth", random.randint, 5, 20)
toolbox.register("xgb_lr", random.uniform, 0.01, 0.3)

# Fusion Weights Genes (W_DT, W_RF, W_XGB)
toolbox.register("w_dt", random.uniform, 0.0, 1.0)
toolbox.register("w_rf", random.uniform, 0.0, 1.0)
toolbox.register("w_xgb", random.uniform, 0.0, 1.0)

# Create Chromosome (Individual) and Population
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.dt_depth, toolbox.rf_est, toolbox.rf_depth,
                  toolbox.xgb_est, toolbox.xgb_depth, toolbox.xgb_lr,
                  toolbox.w_dt, toolbox.w_rf, toolbox.w_xgb), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

print("✅ DNA Configuration Complete. GA is ready to optimize the 9-parameter Fusion Space.")

In [ ]:
# Phase 4.2: Fitness Evaluation
X_train_ga, X_val_ga, y_train_ga, y_val_ga = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE
)

def evaluate_fusion_dna(individual):
    dt_d, rf_e, rf_d, xgb_e, xgb_d, xgb_lr, w1, w2, w3 = individual
    # Clamp to valid integer ranges
    dt_d  = max(2, int(dt_d));   rf_e  = max(10, int(rf_e))
    rf_d  = max(2, int(rf_d));   xgb_e = max(10, int(xgb_e))
    xgb_d = max(2, int(xgb_d))
    # FIXED: VotingClassifier needs integer weights
    w_total = (w1 + w2 + w3) or 3.0
    iw = [max(1, int(round(w / w_total * 10))) for w in [w1, w2, w3]]

    clf1 = DecisionTreeClassifier(max_depth=dt_d, random_state=RANDOM_STATE)
    clf2 = RandomForestClassifier(n_estimators=rf_e, max_depth=rf_d,
                                   random_state=RANDOM_STATE, n_jobs=-1)
    clf3 = xgb.XGBClassifier(n_estimators=xgb_e, max_depth=xgb_d,
                               learning_rate=float(xgb_lr),
                               use_label_encoder=False, eval_metric='mlogloss',
                               random_state=RANDOM_STATE, n_jobs=-1)
    fusion = VotingClassifier(
        estimators=[('dt', clf1), ('rf', clf2), ('xgb', clf3)],
        voting='soft', weights=iw
    )
    try:
        fusion.fit(X_train_ga, y_train_ga)
        return (accuracy_score(y_val_ga, fusion.predict(X_val_ga)),)
    except:
        return (0.0,)

toolbox.register('evaluate', evaluate_fusion_dna)
toolbox.register('mate',   tools.cxBlend, alpha=0.4)
toolbox.register('mutate', tools.mutGaussian, mu=0, sigma=5, indpb=0.3)
toolbox.register('select', tools.selTournament, tournsize=3)
print("GA fitness function registered. Ready to evolve.")

In [ ]:
print('--- STARTING GENETIC ALGORITHM EVOLUTION ---')
pop = toolbox.population(n=12)
NGEN = 3
stats_curve = []

for gen in range(NGEN):
    offspring = algorithms.varAnd(pop, toolbox, cxpb=0.7, mutpb=0.3)
    fits = list(map(toolbox.evaluate, offspring))
    for ind, fit in zip(offspring, fits):
        ind.fitness.values = fit
    pop = toolbox.select(offspring, k=len(pop))
    top_acc = max(ind.fitness.values[0] for ind in pop)
    stats_curve.append(top_acc)
    print("  Gen", gen+1, "/", NGEN, "  best accuracy:", round(top_acc, 4))

best_individual = tools.selBest(pop, k=1)[0]
best_genes      = list(best_individual)
print("Best genes:", [round(g, 3) for g in best_genes])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, NGEN+1), [s*100 for s in stats_curve], 'o-', color='#27ae60', linewidth=2)
ax.set_xlabel('Generation'); ax.set_ylabel('Best Accuracy (%)')
ax.set_title('Figure 7: GA Convergence Curve', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/Figure7_GA_Convergence.png', dpi=300)
plt.show()

In [ ]:
print('--- LINGUISTIC INFERENCE OF BEST CONFIGURATION ---')
_s0 = X_test.iloc[0]
print("Sample: N=", _s0["n"], " P=", _s0["p"], " K=", _s0["k"], " Rain=", _s0["rainfall"])
w1, w2, w3 = best_genes[6], best_genes[7], best_genes[8]
dominant = 'XGBoost' if w3>w1 and w3>w2 else 'Random Forest' if w2>w1 else 'Decision Tree'
print("Dominant model in fusion:", dominant)
print("Weights  DT=", round(w1,2), " RF=", round(w2,2), " XGB=", round(w3,2))

## Phase 5 · GA-Fusion Ensemble — Final Deployment & Validation

In [ ]:
d1, e2, d2, e3, d3, lr3, w1, w2, w3 = best_genes
d1 = max(2,int(d1)); e2 = max(10,int(e2)); d2 = max(2,int(d2))
e3 = max(10,int(e3)); d3 = max(2,int(d3))
w_total = (w1+w2+w3) or 3.0
iw = [max(1, int(round(w/w_total*10))) for w in [w1,w2,w3]]

clf1 = DecisionTreeClassifier(max_depth=d1, random_state=RANDOM_STATE)
clf2 = RandomForestClassifier(n_estimators=e2, max_depth=d2,
                               random_state=RANDOM_STATE, n_jobs=-1)
clf3 = xgb.XGBClassifier(n_estimators=e3, max_depth=d3, learning_rate=float(lr3),
                           use_label_encoder=False, eval_metric='mlogloss',
                           random_state=RANDOM_STATE, n_jobs=-1)

champion_model = VotingClassifier(
    estimators=[('dt',clf1),('rf',clf2),('xgb',clf3)],
    voting='soft', weights=iw
)
champion_model.fit(X_train, y_train)
final_preds    = champion_model.predict(X_test)
final_accuracy = accuracy_score(y_test, final_preds)
print("Champion GA-Fusion accuracy:", round(final_accuracy*100, 2), "%")

In [ ]:
# Accuracy comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))
models = ['DT Baseline','RF Baseline','XGB Baseline','GA-Fusion Ensemble']
accs   = [dt_acc*100, rf_acc*100, xgb_acc*100, final_accuracy*100]
colors = ['#95a5a6','#95a5a6','#95a5a6','#27ae60']
bars = ax.bar(models, accs, color=colors, edgecolor='white', linewidth=1.2)
ax.bar_label(bars, fmt='%.2f%%', padding=3, fontweight='bold')
ax.set_ylim(0, 107); ax.set_ylabel('Accuracy (%)')
ax.set_title('Figure 8: Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/Figure8_Model_Comparison.png', dpi=300)
plt.show()

In [ ]:
# Final confusion matrix
fig, ax = plt.subplots(figsize=(16, 14))
cm_final = confusion_matrix(y_test, final_preds)
sns.heatmap(cm_final, annot=False, cmap='Greens',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Figure 9: GA-Fusion Final Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Crop'); ax.set_ylabel('Actual Crop')
plt.tight_layout()
plt.savefig('../results/Figure9_Final_CM.png', dpi=300)
plt.show()

In [ ]:
print('=== CLASSIFICATION REPORT ===')
report_df = pd.DataFrame(
    classification_report(y_test, final_preds, target_names=le.classes_, output_dict=True)
).T.round(4)
display(report_df)

In [ ]:
# Multi-class ROC curve (one-vs-rest)
y_test_bin = label_binarize(y_test, classes=range(len(le.classes_)))
y_score    = champion_model.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(10, 8))
auc_vals = []
for i in range(len(le.classes_)):
    fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    auc_i = auc(fpr_i, tpr_i)
    auc_vals.append(auc_i)
    ax.plot(fpr_i, tpr_i, lw=0.8, alpha=0.6, label=le.classes_[i] + ' (' + str(round(auc_i,2)) + ')')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_title('Figure 10: Multi-Class ROC  (mean AUC=' + str(round(np.mean(auc_vals),4)) + ')',
             fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', bbox_to_anchor=(1.38,0), fontsize=7)
plt.tight_layout()
plt.savefig('../results/Figure10_ROC.png', dpi=300)
plt.show()

In [ ]:
def get_hybrid_insight(sample_index):
    row           = X_test.iloc[sample_index]
    actual_crop   = le.inverse_transform([y_test[sample_index]])[0]
    predicted_crop = le.inverse_transform([final_preds[sample_index]])[0]
    advice = []
    if row['n'] > 110:    advice.append('N High -> reduce nitrogenous fertiliser')
    elif row['n'] < 40:   advice.append('N Low  -> apply urea or compost')
    if row['ph'] < 6.0:   advice.append('Acidic soil -> apply agricultural lime')
    elif row['ph'] > 7.5: advice.append('Alkaline soil -> apply sulphur')
    if row['rainfall'] < 50:   advice.append('Low rainfall -> supplement irrigation')
    elif row['rainfall'] > 200: advice.append('High rainfall -> ensure drainage')
    return predicted_crop, actual_crop, advice or ['Soil chemistry optimal']

idx = random.randint(0, len(X_test)-1)
pred_c, actual_c, adv = get_hybrid_insight(idx)
print("Sample index :", idx)
print("Actual crop  :", actual_c.upper())
print("Predicted    :", pred_c.upper(), " | Match:", pred_c==actual_c)
print("Fuzzy advice :")
for tip in adv: print('  -', tip)

## Phase 6 · Explainable AI

### 6.1 SHAP — Global Feature Importance

SHAP values show **which features** matter most globally, and how they affect predictions.

In [ ]:
# FIXED: get XGBoost sub-model by name (not fragile index)
xgb_sub        = dict(champion_model.named_estimators_)['xgb']
shap_explainer = shap.TreeExplainer(xgb_sub)
shap_raw       = shap_explainer.shap_values(X_test)

# FIXED: multi-class XGBoost returns LIST of 2D arrays; stack to 3D
if isinstance(shap_raw, list):
    shap_3d = np.stack(shap_raw, axis=2)   # (n_samples, n_features, n_classes)
else:
    shap_3d = shap_raw

# Global importance: mean |SHAP| across samples and classes
global_imp = np.abs(shap_3d).mean(axis=(0, 2))
imp_df = pd.DataFrame({'feature': X_test.columns, 'importance': global_imp})
imp_df = imp_df.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=imp_df, x='importance', y='feature', palette='Blues_r', ax=ax)
ax.set_title('Figure 11: SHAP Global Feature Importance', fontsize=14, fontweight='bold')
ax.set_xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.savefig('../results/Figure11_SHAP_Global.png', dpi=300)
plt.show()

print("Top features:")
display(imp_df.head())

### 6.2 SHAP Beeswarm Plot

Shows the **distribution and direction** of SHAP values for each feature.

In [ ]:
shap.summary_plot(shap_3d[:,:,0], X_test, feature_names=X_test.columns, show=False)
plt.title('Figure 12: SHAP Beeswarm (representative class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/Figure12_SHAP_Beeswarm.png', dpi=300)
plt.show()

### 6.3 LIME — Local Prediction Explanation

LIME explains **why a single prediction** was made for a specific sample.

In [ ]:
# FIXED: single clean LIME initialisation (all duplicate cells merged into one)
lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data = np.array(X_train),
    feature_names = X_train.columns.tolist(),
    class_names   = le.classes_.tolist(),
    mode          = 'classification'
)

lime_idx = 15
lime_exp = lime_explainer.explain_instance(
    data_row   = X_test.iloc[lime_idx],
    predict_fn = champion_model.predict_proba,
    num_features = 9
)
pred_class = le.classes_[final_preds[lime_idx]]
print("LIME explanation for sample", lime_idx, "predicted:", pred_class)
lime_exp.as_pyplot_figure()
plt.title('Figure 13: LIME — ' + pred_class, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/Figure13_LIME.png', dpi=300)
plt.show()

# Interactive in-notebook dashboard
lime_exp.show_in_notebook(show_table=True, show_all=False)

## Phase 7 · Interactive User Prediction

Change the values below and re-run this cell to get a personalised crop recommendation.

In [ ]:
# ── USER INPUTS ──────────────────────────────────────────────────────────
N_val   = 95.0   # Nitrogen   (kg/ha)
P_val   = 45.0   # Phosphorus (kg/ha)
K_val   = 45.0   # Potassium  (kg/ha)
T_val   = 24.5   # Temperature (deg C)
PH_val  = 6.2    # Soil pH (0-14)
R_val   = 210.5  # Rainfall (mm)
# ─────────────────────────────────────────────────────────────────────────

# FIXED: compute engineered features from actual user inputs (not training means)
input_dict = {
    'n':            N_val,
    'p':            P_val,
    'k':            K_val,
    'ph':           PH_val,
    'rainfall':     R_val,
    'temperature':  T_val,
    'n_p_ratio':    N_val / (P_val + 1),
    'n_k_ratio':    N_val / (K_val + 1),
    'temp_rainfall': T_val * R_val
}
input_df = pd.DataFrame([input_dict])[X_train.columns]  # enforce column order

# ML Layer
probs      = champion_model.predict_proba(input_df)[0]
pred_idx   = np.argmax(probs)
crop_name  = le.classes_[pred_idx]
confidence = probs[pred_idx] * 100
top3 = [(le.classes_[i], round(probs[i]*100,2)) for i in np.argsort(probs)[::-1][:3]]

# Fuzzy Layer
try:
    crop_simulator.input['Nitrogen']    = N_val
    crop_simulator.input['Phosphorus']  = P_val
    crop_simulator.input['Potassium']   = K_val
    crop_simulator.input['pH']          = PH_val
    crop_simulator.input['Rainfall']    = R_val
    crop_simulator.input['Temperature'] = T_val
    crop_simulator.compute()
    # FIXED: clamp using actual class count, not hardcoded 39
    fuzzy_idx  = int(round(crop_simulator.output['Crop']))
    fuzzy_idx  = max(0, min(len(le.classes_)-1, fuzzy_idx))
    fuzzy_crop = le.classes_[fuzzy_idx]
except Exception as e:
    fuzzy_crop = '(FIS unavailable: ' + str(e) + ')'

# Agronomic advice
advice = []
if N_val > 110:   advice.append('N High -> reduce nitrogenous fertiliser')
elif N_val < 40:  advice.append('N Low  -> apply urea or compost')
if PH_val < 6.0:  advice.append('Acidic soil -> apply lime')
elif PH_val > 7.5: advice.append('Alkaline soil -> apply sulphur')
if R_val < 50:    advice.append('Low rainfall -> irrigate')
elif R_val > 200: advice.append('High rainfall -> ensure drainage')
if not advice: advice = ['Soil chemistry appears optimal']

# LIME local explanation
user_exp = lime_explainer.explain_instance(
    data_row   = input_df.iloc[0],
    predict_fn = champion_model.predict_proba,
    num_features = 9
)

print('=' * 52)
print("  ML RECOMMENDED CROP :", crop_name.upper())
print("  ML CONFIDENCE       :", round(confidence,2), "%")
print("  FUZZY PREDICTION    :", fuzzy_crop)
print('  TOP-3 PREDICTIONS   :')
for crop_t, prob_t in top3:
    print("   ", crop_t.ljust(20), str(prob_t) + "%")
print('  AGRONOMIC ADVICE    :')
for tip in advice: print('    -', tip)
print('=' * 52)

user_exp.show_in_notebook(show_table=True, show_all=False)